# 02 — ST Population Sweep

Generates the SPEC §7.3 deliverables for the ST (`WindowedSpatioTemporalGATNet`) architecture across the full matrix:

- **3 regimes:** `kfold-5`, `kfold-10`, `loso`
- **2 mt:** 2, 4
- **2 paths:**
  - `native` — built-in `model.explain()` with reductions (heads → layers → α_k weighting → symmetric pair matrix → row-sum). SPEC §6.1 primary path. Essentially free at inference.
  - `supplementary` — GNNExplainer with `node_mask_type='object'` (mask is `[23]`, not `[23, 326]`). SPEC §6.4 cross-check.

= **12 PopulationResults** total (6 native + 6 supplementary). Each lands at `research/xai/st/{regime}/mt{N}/{path}/` with the §7.3 deliverables, including ST-only `temporal_attention.csv`.

> **Runtime note.** Native attention is essentially free (~30 s/regime). Supplementary GNNExplainer ≈ 1 s/trial at `epochs=200` on GPU. `loso × supplementary` ≈ ~5–10 min. Set `run_supplementary=False` in any cell to skip the GNN object-mask path.

> **PyG 2.7 quirk handled in the supplementary path:** cuDNN's GRU backward refuses eval-mode gradients, so the explainer call is wrapped in `torch.backends.cudnn.flags(enabled=False)`. See `src/xai/st_explainer.py:explain_supplementary_checkpoint`.

Reference: [`docs/SPEC_xai_graph.md`](../../../docs/SPEC_xai_graph.md) (rev. 4).

In [1]:
%matplotlib inline
import os, sys
from pathlib import Path

import numpy as np
import torch
from scipy import stats

PROJECT_ROOT = Path(os.getcwd()).resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.xai import (
    XAIRunConfig,
    run_st, run_st_supplementary,
    plot_montage_channel_importance, plot_pair_matrix, plot_temporal_attention,
    write_run_json,
    PopulationResult,
    CHANNEL_NAMES, N_CH,
)

ST_EXPERIMENT_ROOT = PROJECT_ROOT / "research/experiments/20260501/spatial_temporal_graph"
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"PROJECT_ROOT       = {PROJECT_ROOT}")
print(f"ST_EXPERIMENT_ROOT exists: {ST_EXPERIMENT_ROOT.is_dir()}")
print(f"DEVICE             = {DEVICE}")

PROJECT_ROOT       = /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method
ST_EXPERIMENT_ROOT exists: True
DEVICE             = cuda:0


## Helper

`run_st_cell(regime, mt, run_supplementary, gnn_epochs)` runs the native attention path
(always) and the supplementary GNN object-mask path (when `run_supplementary=True`)
for one `(regime, mt)` matrix cell, persists everything to disk, and returns
`{path → PopulationResult}`.

In [2]:
def run_st_cell(
    regime: str,
    mt: int,
    run_supplementary: bool = True,
    gnn_epochs: int = 200,
    gnn_lr: float = 0.01,
    head_reduce: str = "mean",
    layer_reduce: str = "mean",
) -> dict:
    """Run native + (optional) supplementary paths for one (regime, mt) cell."""
    out: dict = {}

    # --- native attention (always) ---------------------------------------
    native_dir = PROJECT_ROOT / f"research/xai/st/{regime}/mt{mt}/native"
    native_dir.mkdir(parents=True, exist_ok=True)
    cfg_native = XAIRunConfig(
        arch="st", hb="hbo", regime=regime, mt=mt,
        experiment_root=str(ST_EXPERIMENT_ROOT),
        output_dir=str(native_dir),
        device=DEVICE,
        head_reduce=head_reduce,
        layer_reduce=layer_reduce,
        seed=42,
    )
    print(f"[{regime} mt={mt}] {'native':>13s} → {native_dir.relative_to(PROJECT_ROOT)} ...")
    res_native = run_st(cfg_native)
    res_native.to_csv(native_dir)
    write_run_json(cfg_native, res_native, native_dir)
    plot_montage_channel_importance(res_native, native_dir)
    plot_pair_matrix(res_native, native_dir)
    plot_temporal_attention(res_native, native_dir)
    out["native"] = res_native
    top5 = [CHANNEL_NAMES[i] for i in (-res_native.channel_importance_mean).argsort()[:5]]
    print(f"               done — n_trials={res_native.n_trials}/{res_native.n_trials_total} "
          f"({res_native.included_pct:.1f}%), top-5: {top5}")

    if run_supplementary:
        sup_dir = PROJECT_ROOT / f"research/xai/st/{regime}/mt{mt}/supplementary"
        sup_dir.mkdir(parents=True, exist_ok=True)
        cfg_sup = XAIRunConfig(
            arch="st", hb="hbo", regime=regime, mt=mt,
            experiment_root=str(ST_EXPERIMENT_ROOT),
            output_dir=str(sup_dir),
            device=DEVICE,
            gnn_explainer_epochs=gnn_epochs,
            gnn_explainer_lr=gnn_lr,
            run_supplementary_gnnexplainer=True,
            seed=42,
        )
        print(f"[{regime} mt={mt}] {'supplementary':>13s} → {sup_dir.relative_to(PROJECT_ROOT)} ...")
        res_sup = run_st_supplementary(cfg_sup)
        res_sup.to_csv(sup_dir)
        write_run_json(cfg_sup, res_sup, sup_dir)
        plot_montage_channel_importance(res_sup, sup_dir)
        plot_pair_matrix(res_sup, sup_dir)
        out["supplementary"] = res_sup
        top5 = [CHANNEL_NAMES[i] for i in (-res_sup.channel_importance_mean).argsort()[:5]]
        print(f"               done — top-5: {top5}")

    return out


def report_c5_agreement(results: dict, label: str) -> None:
    """SPEC §11 C5: Spearman ρ between native and supplementary top-10 channel rankings ≥ 0.4."""
    if "supplementary" not in results:
        print(f"[{label}] C5 skipped — supplementary path not run")
        return
    a = results["native"].channel_importance_mean
    b = results["supplementary"].channel_importance_mean
    ra = (-a).argsort().argsort()
    rb = (-b).argsort().argsort()
    rho, _ = stats.spearmanr(ra, rb)
    flag = "✓" if rho >= 0.4 else "✗"
    print(f"[{label}] C5 — ρ(native top-10, supplementary top-10) = {rho:+.3f}   {flag} (≥ 0.4)")

## kfold-5 × mt=2

Native ≈ 30 s. Supplementary ≈ 3–5 min. ~5 min total.

In [3]:
st_kfold5_mt2 = run_st_cell(regime="kfold-5", mt=2)
report_c5_agreement(st_kfold5_mt2, "kfold-5 × mt=2")

[kfold-5 mt=2]        native → research/xai/st/kfold-5/mt2/native ...
               done — n_trials=94/124 (75.8%), top-5: ['S8_D5', 'S7_D6', 'S8_D7', 'S5_D5', 'S7_D7']
[kfold-5 mt=2] supplementary → research/xai/st/kfold-5/mt2/supplementary ...
               done — top-5: ['S6_D3', 'S2_D1', 'S1_D3', 'S7_D6', 'S8_D5']
[kfold-5 × mt=2] C5 — ρ(native top-10, supplementary top-10) = +0.067   ✗ (≥ 0.4)


## kfold-5 × mt=4

Native ≈ 1 min. Supplementary ≈ 6–10 min.

In [4]:
st_kfold5_mt4 = run_st_cell(regime="kfold-5", mt=4)
report_c5_agreement(st_kfold5_mt4, "kfold-5 × mt=4")

[kfold-5 mt=4]        native → research/xai/st/kfold-5/mt4/native ...
               done — n_trials=177/248 (71.4%), top-5: ['S8_D5', 'S7_D7', 'S7_D6', 'S5_D5', 'S3_D1']
[kfold-5 mt=4] supplementary → research/xai/st/kfold-5/mt4/supplementary ...
               done — top-5: ['S1_D3', 'S3_D6', 'S4_D7', 'S3_D1', 'S8_D5']
[kfold-5 × mt=4] C5 — ρ(native top-10, supplementary top-10) = +0.212   ✗ (≥ 0.4)


## kfold-10 × mt=2

Similar to kfold-5 mt2 in total runtime (same number of trials, more folds).

In [5]:
st_kfold10_mt2 = run_st_cell(regime="kfold-10", mt=2)
report_c5_agreement(st_kfold10_mt2, "kfold-10 × mt=2")

[kfold-10 mt=2]        native → research/xai/st/kfold-10/mt2/native ...
               done — n_trials=97/124 (78.2%), top-5: ['S8_D5', 'S4_D4', 'S5_D5', 'S1_D1', 'S8_D7']
[kfold-10 mt=2] supplementary → research/xai/st/kfold-10/mt2/supplementary ...
               done — top-5: ['S1_D3', 'S6_D3', 'S8_D5', 'S3_D1', 'S3_D6']
[kfold-10 × mt=2] C5 — ρ(native top-10, supplementary top-10) = -0.091   ✗ (≥ 0.4)


## kfold-10 × mt=4

Native ≈ 1 min. Supplementary ≈ 10–15 min.

In [6]:
st_kfold10_mt4 = run_st_cell(regime="kfold-10", mt=4)
report_c5_agreement(st_kfold10_mt4, "kfold-10 × mt=4")

[kfold-10 mt=4]        native → research/xai/st/kfold-10/mt4/native ...
               done — n_trials=186/248 (75.0%), top-5: ['S5_D5', 'S3_D1', 'S7_D7', 'S8_D7', 'S5_D8']
[kfold-10 mt=4] supplementary → research/xai/st/kfold-10/mt4/supplementary ...
               done — top-5: ['S1_D3', 'S6_D3', 'S8_D5', 'S3_D6', 'S3_D1']
[kfold-10 × mt=4] C5 — ρ(native top-10, supplementary top-10) = +0.144   ✗ (≥ 0.4)


## LOSO × mt=2

⚠ **Long-running supplementary.** 62 subjects × ~2 val trials × 200 epochs ≈ 5–10 min on GPU. Native is fast (~1 min).

In [7]:
st_loso_mt2 = run_st_cell(regime="loso", mt=2)
report_c5_agreement(st_loso_mt2, "loso × mt=2")

[loso mt=2]        native → research/xai/st/loso/mt2/native ...
               done — n_trials=88/124 (71.0%), top-5: ['S4_D4', 'S8_D8', 'S5_D5', 'S7_D4', 'S7_D6']
[loso mt=2] supplementary → research/xai/st/loso/mt2/supplementary ...
               done — top-5: ['S1_D3', 'S6_D3', 'S8_D5', 'S7_D6', 'S2_D1']
[loso × mt=2] C5 — ρ(native top-10, supplementary top-10) = -0.036   ✗ (≥ 0.4)


## LOSO × mt=4

⚠ **Longest.** 62 subjects × ~4 val trials. Supplementary ≈ 15–25 min.

In [8]:
st_loso_mt4 = run_st_cell(regime="loso", mt=4)
report_c5_agreement(st_loso_mt4, "loso × mt=4")

[loso mt=4]        native → research/xai/st/loso/mt4/native ...
               done — n_trials=179/248 (72.2%), top-5: ['S5_D5', 'S7_D6', 'S8_D5', 'S5_D8', 'S7_D7']
[loso mt=4] supplementary → research/xai/st/loso/mt4/supplementary ...
               done — top-5: ['S1_D3', 'S6_D3', 'S8_D5', 'S3_D6', 'S7_D6']
[loso × mt=4] C5 — ρ(native top-10, supplementary top-10) = +0.187   ✗ (≥ 0.4)


## Summary

Top-5 channels per cell, mt=2 ↔ mt=4 stability per path (SPEC §11 C3 ≥ 0.5), and a quick look at the temporal attention concentration (peak window per regime/mt).

In [9]:
ALL_RUNS = {
    "kfold-5/mt2":  st_kfold5_mt2,
    "kfold-5/mt4":  st_kfold5_mt4,
    "kfold-10/mt2": st_kfold10_mt2,
    "kfold-10/mt4": st_kfold10_mt4,
    "loso/mt2":     st_loso_mt2,
    "loso/mt4":     st_loso_mt4,
}

print("=" * 78)
print("Top-5 channels per (regime, mt, path)")
print("=" * 78)
for cell, res in ALL_RUNS.items():
    for path, r in res.items():
        top5 = [CHANNEL_NAMES[i] for i in (-r.channel_importance_mean).argsort()[:5]]
        print(f"  {cell:>12s} | {path:>13s} : {top5}")

print()
print("=" * 78)
print("SPEC §11 C3 — mt=2 vs mt=4 stability per regime/path (ρ ≥ 0.5)")
print("=" * 78)
for regime in ("kfold-5", "kfold-10", "loso"):
    mt2 = ALL_RUNS[f"{regime}/mt2"]
    mt4 = ALL_RUNS[f"{regime}/mt4"]
    for path in ("native", "supplementary"):
        if path not in mt2 or path not in mt4:
            continue
        r2 = (-mt2[path].channel_importance_mean).argsort().argsort()
        r4 = (-mt4[path].channel_importance_mean).argsort().argsort()
        rho, _ = stats.spearmanr(r2, r4)
        flag = "✓" if rho >= 0.5 else "✗"
        print(f"  {regime:>9s} {path:>13s}: ρ(mt2, mt4) = {rho:+.3f}   {flag}")

print()
print("=" * 78)
print("Temporal attention — peak window per (regime, mt)")
print("=" * 78)
for cell, res in ALL_RUNS.items():
    if "native" not in res:
        continue
    r = res["native"]
    peak_idx = int(np.argmax(r.temporal_attention_mean))
    if r.window_times is not None:
        t_start, t_end = r.window_times[peak_idx]
        print(f"  {cell:>12s} : peak window k={peak_idx} → [{t_start:.1f}–{t_end:.1f}] s, "
              f"α={r.temporal_attention_mean[peak_idx]:.3f}")
    else:
        print(f"  {cell:>12s} : peak window k={peak_idx}, α={r.temporal_attention_mean[peak_idx]:.3f}")

Top-5 channels per (regime, mt, path)
   kfold-5/mt2 |        native : ['S8_D5', 'S7_D6', 'S8_D7', 'S5_D5', 'S7_D7']
   kfold-5/mt2 | supplementary : ['S6_D3', 'S2_D1', 'S1_D3', 'S7_D6', 'S8_D5']
   kfold-5/mt4 |        native : ['S8_D5', 'S7_D7', 'S7_D6', 'S5_D5', 'S3_D1']
   kfold-5/mt4 | supplementary : ['S1_D3', 'S3_D6', 'S4_D7', 'S3_D1', 'S8_D5']
  kfold-10/mt2 |        native : ['S8_D5', 'S4_D4', 'S5_D5', 'S1_D1', 'S8_D7']
  kfold-10/mt2 | supplementary : ['S1_D3', 'S6_D3', 'S8_D5', 'S3_D1', 'S3_D6']
  kfold-10/mt4 |        native : ['S5_D5', 'S3_D1', 'S7_D7', 'S8_D7', 'S5_D8']
  kfold-10/mt4 | supplementary : ['S1_D3', 'S6_D3', 'S8_D5', 'S3_D6', 'S3_D1']
      loso/mt2 |        native : ['S4_D4', 'S8_D8', 'S5_D5', 'S7_D4', 'S7_D6']
      loso/mt2 | supplementary : ['S1_D3', 'S6_D3', 'S8_D5', 'S7_D6', 'S2_D1']
      loso/mt4 |        native : ['S5_D5', 'S7_D6', 'S8_D5', 'S5_D8', 'S7_D7']
      loso/mt4 | supplementary : ['S1_D3', 'S6_D3', 'S8_D5', 'S3_D6', 'S7_D6']

SPEC §11 C3 —

## Done

Outputs are persisted to `research/xai/st/{regime}/mt{N}/{native|supplementary}/` and ready for `03_cross_arch_comparison.ipynb`. The C5 (native vs supplementary cross-check) and C3 (mt stability) acceptance checks are reported above; C2 (across-fold), C4 (3-way for SG), and C6 (biological-prior overlap) live in `03_cross_arch_comparison.ipynb`.